# 01b Subword Tokenization (BPE, WordPiece, SentencePiece) & Embedding Mechanics

Implementing Byte-Pair Encoding (BPE) from scratch, tiktoken execution, and PyTorch Embedding lookup matrices.

## Technique 1: BPE Tokenizer Algorithm Implementation from Scratch

In [ ]:
# Byte-Pair Encoding (BPE) Tokenizer Implementation from Scratch
import re
from collections import Counter, defaultdict

def get_stats(vocab: dict) -> dict:
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_vocab(pair: tuple, v_in: dict) -> dict:
    v_out = {}
    bigram = re.escape(' '.join(pair))
    pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = pattern.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

# Sample Corpus Vocabulary (Character Level Initial State)
vocab = {
    'l o w </w>': 5,
    'l o w e r </w>': 2,
    'n e w e s t </w>': 6,
    'w i d e s t </w>': 3
}

num_merges = 5
for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best_pair = max(pairs, key=pairs.get)
    vocab = merge_vocab(best_pair, vocab)
    print(f"Merge #{i+1}: {best_pair} -> Vocabulary State: {vocab}")

## Technique 2: Modern BPE Tokenization with `tiktoken`

In [ ]:
# Using OpenAI Tiktoken and HuggingFace Tokenizers
print("=== Tokenization Execution with tiktoken & HuggingFace ===")
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base") # GPT-4 / GPT-3.5-turbo tokenizer
    tokens = enc.encode("FlashAttention optimizes KV cache memory in vLLM.")
    print("tiktoken Token IDs:", tokens)
    print("tiktoken Decoded Tokens:", [enc.decode([t]) for t in tokens])
except Exception as e:
    print("tiktoken execution skipped:", e)

## Technique 3: PyTorch Token Embedding Lookup Matrix ($E \in \mathbb{R}^{|V| \times d}$)

In [ ]:
import torch
import torch.nn as nn

# Embedding Lookup Matrix Algebra
vocab_size = 1000
d_model = 64

embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)
input_ids = torch.tensor([12, 45, 892, 3]) # Batch of 4 token IDs

# Matrix lookup E[input_ids]
token_vectors = embedding_layer(input_ids)

print(f"Input Token IDs Shape: {input_ids.shape}")
print(f"Output Token Vector Tensor Shape: {token_vectors.shape}")
print(f"Norm of First Token Embedding: {token_vectors[0].norm().item():.3f}")